# ST Guitar — Stage 7G-E3-S0 Scientific Diagnostic

Amaç: frozen R2 monolitik `OPEN_LOW vs COMPACT` MLP'nin ultra-quality hedefini neden geçemediğini bilimsel olarak teşhis etmek.

**Değişmeyenler:** 399 development Teacher-GOLD, 40 family, 40 feature, aynı `[32,16]` MLP, LR `0.001`, 60 epoch, threshold `0.5`.

**Yasak:** scheduler/tuning/yeni feature/specialist training/early stopping/best-epoch checkpoint/E3-E/Stage7E/checkpoint/production.

Specialist mimari yalnız **TARGET ARCHITECTURE CANDIDATE** olarak kalır.


In [ ]:
PINNED_CODE_SHA="e477fe75de101f604f8d160ab446e16f00b07537"
ANIMETAB_COMMIT="18c0993cbe0a0948cbf0b7768bcb09ff81c23a9a"
EXPECTED_CHOICES_SHA256="db0e752ec7b9e0e1b333a217d904175f4e57cd89a32b2511330ebab7b8c6c12e"
EXPECTED_NUMPY="2.4.6"; EXPECTED_SCIPY="1.17.1"; EXPECTED_SKLEARN="1.9.0"

import os,sys,subprocess,importlib
os.chdir("/content")
subprocess.run(["rm","-rf","st-guitar-fingering-training"],check=True)
subprocess.run(["git","clone","-q","https://github.com/khfy7wpr5p-maker/st-guitar-fingering-training.git"],check=True)
os.chdir("/content/st-guitar-fingering-training")
subprocess.run(["git","checkout","-q",PINNED_CODE_SHA],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q",f"numpy=={EXPECTED_NUMPY}",f"scipy=={EXPECTED_SCIPY}",f"scikit-learn=={EXPECTED_SKLEARN}","joblib==1.5.3","threadpoolctl==3.6.0"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e","."],check=True)
sys.path.insert(0,"/content/st-guitar-fingering-training/src"); importlib.invalidate_caches()
import numpy as np, scipy, sklearn
assert (np.__version__,scipy.__version__,sklearn.__version__)==(EXPECTED_NUMPY,EXPECTED_SCIPY,EXPECTED_SKLEARN)
assert subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()==PINNED_CODE_SHA
print("===== S0 SETUP PASS =====")
print("Python",sys.version.split()[0],"NumPy",np.__version__,"SciPy",scipy.__version__,"sklearn",sklearn.__version__)
SETUP_READY=True


In [ ]:
assert SETUP_READY
from google.colab import files
from pathlib import Path
import hashlib,json
uploaded=files.upload()
assert len(uploaded)==1,"STOP: yalnız 400-of-400 JSON yükle."
name=next(iter(uploaded)); data=uploaded[name]
actual_choices_sha=hashlib.sha256(data).hexdigest()
assert actual_choices_sha==EXPECTED_CHOICES_SHA256,(actual_choices_sha,EXPECTED_CHOICES_SHA256)
choices_path=Path("/content/st-guitar-fingering-training/ST_Guitar_E3_Batch01_choices_400of400.json")
choices_path.write_bytes(data)
print("===== CHOICES SHA PASS =====",actual_choices_sha)
CHOICES_READY=True


In [ ]:
assert CHOICES_READY
import platform,urllib.parse,urllib.request
from tempfile import TemporaryDirectory
from st_guitar_fingering_training.target_free_musicxml import parse_target_free_musicxml
from st_guitar_fingering_training.stage7g_e3_e_a3 import reconstruct_frozen_open_low_compact_specialists
from st_guitar_fingering_training.stage7g_e3_r2_learning import build_stage7g_e3_r2_disagreement_pool,rows_from_choices
from st_guitar_fingering_training.stage7g_e3_s0_diagnostic import STAGE7G_E3_S0_CONFIG

protocol=json.loads(Path("evidence/stage7g_e3_s0_failure_diagnostic_protocol.json").read_text())
assert protocol["status"]=="PREREGISTERED_NO_RESULTS"
assert protocol["architecture_decision"]["specialist_architecture_status"]=="TARGET_ARCHITECTURE_CANDIDATE_ONLY"
manifest=json.loads(Path("evidence/stage7g_c_r1_animetab_batch01_manifest.json").read_text())
assert manifest["family_count"]==40 and manifest["staff_id"]=="2" and manifest["part_id"]=="P1"
sources=[]
with TemporaryDirectory() as tmp:
    root=Path(tmp)
    for i,item in enumerate(manifest["sources"],1):
        url=("https://raw.githubusercontent.com/amamiya-yuuko/AnimeTAB/"+ANIMETAB_COMMIT+"/AnimeTAB/Entire%20songs/"+urllib.parse.quote(item["filename"],safe=""))
        req=urllib.request.Request(url,headers={"User-Agent":"st-guitar-stage7g-e3-s0-colab-v1"})
        with urllib.request.urlopen(req,timeout=45) as r: raw=r.read()
        assert hashlib.sha256(raw).hexdigest()==item["sha256"]
        p=root/f"{i:03d}.xml"; p.write_bytes(raw)
        sources.append(parse_target_free_musicxml(p,family_id=item["family_id"],tuning=manifest["tuning_midi"],pitch_mode=manifest["pitch_mode"],part_id=manifest["part_id"],staff_id=manifest["staff_id"]))
models,guard=reconstruct_frozen_open_low_compact_specialists()
assert guard["status"]=="PASS_STAGE7B_C2_OPEN_LOW_COMPACT_RECONSTRUCTION"
pool=build_stage7g_e3_r2_disagreement_pool(tuple(sources),specialist_models=models)
rows,preflight=rows_from_choices(pool,json.loads(choices_path.read_text()))
assert preflight["status"]=="R2_PREFLIGHT_PASS_STOP_BEFORE_MANUAL_TRAIN"
assert (preflight["decisive_rows"],preflight["families"],preflight["feature_count"])==(399,40,40)
identity={"code_sha":PINNED_CODE_SHA,"animetab_commit":ANIMETAB_COMMIT,"choices_sha256":actual_choices_sha,"python":platform.python_version(),"numpy":np.__version__,"scipy":scipy.__version__,"scikit_learn":sklearn.__version__,"s0_config":STAGE7G_E3_S0_CONFIG,"e3e_teacher_gold_used":False,"stage7e_used":False,"checkpoint_retained":False}
print(json.dumps(preflight,indent=2))
print("===== S0 PREFLIGHT PASS — STOP BEFORE MANUAL DIAGNOSTIC =====")
PREFLIGHT_READY=True


# ▶ MANUAL S0 DIAGNOSTIC

Bu hücreyi sen çalıştır. 5 family-isolated fold + 25/50/75/100 family learning curve + family-cluster bootstrap üretir. Minimum Val Loss yalnız diagnostiktir; checkpoint seçmez.


In [ ]:
assert PREFLIGHT_READY
from st_guitar_fingering_training.stage7g_e3_s0_diagnostic import stage7g_e3_s0_diagnostic_report
print("===== S0 START ===== 5-fold scientific diagnostic çalışıyor...")
report=stage7g_e3_s0_diagnostic_report(rows)
def f4(x): return "N/A" if x is None else f"{x:.4f}"
print("\n===== OUTER FOLDS / EPOCH 60 =====")
print(f"{'F':>2} {'N':>4} {'C':>3} {'MinEp':>5} {'MinLoss':>8} {'Loss60':>8} {'MacroF1':>8} {'BalAcc':>8} {'AP':>8} {'Prec':>8} {'Rec':>8}")
for r in report["outer_folds"]:
    m=r["minimum_val_loss"]; q=r["final_epoch"]
    print(f"{r['fold']:2d} {r['validation_rows']:4d} {r['validation_compact_support']:3d} {m['epoch']:5d} {m['value']:8.4f} {q['log_loss']:8.4f} {q['macro_f1']:8.4f} {f4(q['balanced_accuracy']):>8} {f4(q['average_precision']):>8} {q['compact_precision']:8.4f} {f4(q['compact_recall']):>8}")
oof=report["oof_final_epoch"]
print("\n===== 399-ROW OOF =====")
for k in ("accuracy","macro_f1","balanced_accuracy","compact_precision","compact_recall","average_precision","roc_auc","brier_score","mcc"): print(f"{k:22s}: {f4(oof[k])}")
print("TP/FP/FN/TN:",oof["tp"],oof["fp"],oof["fn"],oof["tn"])
print("\n===== FAMILY-BOOTSTRAP 95% CI =====")
for k,v in report["family_cluster_bootstrap_95ci"]["intervals"].items(): print(f"{k:22s}: {f4(v['point_estimate'])} [{f4(v['lower'])}, {f4(v['upper'])}]")
print("\n===== LEARNING CURVE =====")
for r in report["learning_curve"]:
    m=r["oof_metrics"]; print(f"{r['target_fraction']*100:3.0f}% families | MacroF1 {m['macro_f1']:.4f} | BalAcc {f4(m['balanced_accuracy'])} | AP {f4(m['average_precision'])} | CRec {f4(m['compact_recall'])}")
print("\n===== DIAGNOSTIC FLAGS =====")
print(json.dumps(report["diagnostic_flags"],indent=2))
result={"schema":"st-guitar-stage7g-e3-s0-colab-result-v1","stage":"7G-E3-S0","identity":identity,"preflight":preflight,"report":report,"raw_event_rows_exported":False,"checkpoint_retained":False,"specialist_architecture_activated":False,"e3e_teacher_gold_used":False,"stage7e_used":False,"production_or_shadow_integration":False}
out=Path("ST_Guitar_Stage7G_E3_S0_Diagnostic_result.json"); payload=json.dumps(result,indent=2)+"\n"; out.write_text(payload)
print("\n===== S0 COMPLETE =====")
print("Result SHA256:",hashlib.sha256(payload.encode()).hexdigest())
files.download(str(out))
